In [9]:
from sklearn.datasets import fetch_openml
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X, y = fetch_openml(name='credit-g', version=1, return_X_y=True, as_frame = True)

le = LabelEncoder()
y = le.fit_transform(y)

X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.3, random_state=42)

print("Shape of training data ",X_train.shape)
print("Shape of test data : ", X_test.shape)

Shape of training data  (700, 48)
Shape of test data :  (300, 48)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

initial_score = rf.score(X_test, y_test)
print(f"Initial accuracy before tuning : {initial_score:.2f}")

Initial Score before tuning : 0.76


In [16]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators' : [100, 200, 300],
    'max_depth' : [10, 20, 30],
    'min_samples_split' : [2, 5, 10],
    'criterion' : ['gini', 'entropy']
}

In [19]:
grid_search = GridSearchCV(estimator=rf, param_grid = param_grid, cv=3, n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

print(f"Best parameter (Grid Search) {grid_search.best_params_}")
best_rf = grid_search.best_estimator_

#evaluasi data pada data test
grid_search_score = best_rf.score(X_test, y_test)
print(f"Accuracy after grid Search : {grid_search_score:.2f}")

Fitting 3 folds for each of 54 candidates, totalling 162 fits
[CV] END criterion=gini, max_depth=10, min_samples_split=2, n_estimators=100; total time=   0.2s
[CV] END criterion=gini, max_depth=10, min_samples_split=2, n_estimators=100; total time=   0.2s
[CV] END criterion=gini, max_depth=10, min_samples_split=2, n_estimators=100; total time=   0.2s
[CV] END criterion=gini, max_depth=10, min_samples_split=5, n_estimators=100; total time=   0.2s
[CV] END criterion=gini, max_depth=10, min_samples_split=2, n_estimators=200; total time=   0.5s
[CV] END criterion=gini, max_depth=10, min_samples_split=5, n_estimators=100; total time=   0.3s
[CV] END criterion=gini, max_depth=10, min_samples_split=2, n_estimators=200; total time=   0.5s
[CV] END criterion=gini, max_depth=10, min_samples_split=2, n_estimators=200; total time=   0.6s
[CV] END criterion=gini, max_depth=10, min_samples_split=2, n_estimators=300; total time=   0.7s
[CV] END criterion=gini, max_depth=10, min_samples_split=5, n_est

In [21]:
from skopt import BayesSearchCV

param_space = {
    'n_estimators' : (100, 500),
    'max_depth' : (10, 50),
    'min_samples_split' : (2, 10),
    'criterion' : ['gini', 'entropy']
}

bayes_search = BayesSearchCV(estimator=rf, search_spaces=param_space, n_iter=32, cv=3, n_jobs=-1, random_state=42)
bayes_search.fit(X_train, y_train)

print(f"Best Parameters (Bayesian Optimization) : {bayes_search.best_params_}")
best_bayes_rf = bayes_search.best_estimator_

bayes_search_score = best_bayes_rf.score(X_test, y_test)
print(f"Accuracy after Bayesian Optimization : {bayes_search_score}")

Best Parameters (Bayesian Optimization) : OrderedDict([('criterion', 'gini'), ('max_depth', 19), ('min_samples_split', 4), ('n_estimators', 499)])
Accuracy after Bayesian Optimization : 0.77
